# ZL+vMIT Hybrid EOS Notebook

This notebook generates hybrid EOS tables with a hadron-quark phase transition.
- Zero and finite temperature
- Beta equilibrium and fixed Y_C (or equivalently Y_e)

## Physical Content

- **Hadronic phase (H):** ZL model with nucleons (proton, neutron)
- **Quark phase (Q):** vMIT bag model with u, d, s quarks
- **Mixed phase:** Global electric charge neutrality (η=0), Local electric charge neutrality (η=1), or intermediate (0<η<1) construction
- **Equilibrium modes:** β-equilibrium or fixed charge fraction Y_C

## Notebook Structure

1. **Input Parameters** - Define grids and model parameters
2. **Beta Equilibrium** - Compute EOS in beta_eq as a function of (n_B, T, η)
3. **Fixed Y_C** - Compute EOS at fixed Y_C as a function of (n_B, Y_C, T, η)
4. **Plots** - Visualization of results

## Modules

| Module | Description |
|--------|-------------|
| `zlvmit_mixed_phase_eos.py` | Core solvers and table generation |
| `zl_compute_tables.py` | Pure hadronic phase tables |
| `vmit_compute_tables.py` | Pure quark phase tables |
| `zlvmit_table_reader.py` | Load and interpolate EOS tables |
| `zlvmit_plot_results.py` | Plotting utilities |

## Output Files 

- `pure_H_table_*.dat` - Pure hadronic phase table
- `pure_Q_table_*.dat` - Pure quark phase table  
- `table_hybrid_*_result.dat` - Primary output (only conserved charges)
- `table_hybrid_*_complete.dat` - Complete thermodynamic quantities output
- `boundaries_*.dat` - Phase transition boundaries

---

In [ ]:
# =============================================================================
# IMPORT
# =============================================================================

# Import
import numpy as np
import os
from datetime import datetime

from eos.zl.parameters import Parameters as ZLParams
from eos.vmit.parameters import VMITParams, get_vmit_custom
from eos.zlvmit.mixed_phase_eos import (
    get_or_compute_boundaries, 
    generate_unified_table,
    results_to_dict_primary, 
    results_to_dict_complete, 
    save_table_full
)
from eos.zl.table import TableSettings as ZLTableSettings, compute_table as compute_zl_table
from eos.vmit.compute_tables import VMITTableSettings, compute_vmit_table
from eos.zlvmit.plot_results import (
    plot_betaeq, 
    plot_fixedyc, 
    plot_composition, 
    plot_mixed_phase_boundaries,
    setup_matplotlib_style,
    ETA_STYLES,
    N_SAT,
)


import matplotlib.pyplot as plt

# Input Parameters

In [ ]:
# ==============================================================================
# Build grid vectors aligned with SFHO CompOSE (with low-T extension)
# ==============================================================================
from eos.general.compose import ComposeLookup
n0 = 0.16  # Nuclear saturation density
_sfho_grids = ComposeLookup(
    '/Users/mircoguerrini/Desktop/Research/Compose/SFHO_Compose',
    name='SFHO_grids',
)
_sfho_T = _sfho_grids.T_grid

# --- nB_vector: SFHO nB grid, but only points with n_B > 0.016 fm^-3 -----------
n0 = 0.16  # Nuclear saturation density
n_B_min = 0.1 * n0        
n_B_max = 12.0 * n0        
n_B_steps = 300           # Number of density points
n_B_values = np.linspace(n_B_min, n_B_max, n_B_steps)

# --- T_vector: SFHO T grid (0.1 to ~158 MeV) plus extra points 0.01 → 0.1 ------
# _T_extra: log-spaced 0.01 → 0.1 (continuation of SFHO's log step ≈ 1.0965)
_log_ratio = (_sfho_T[-1] / _sfho_T[0]) ** (1.0 / (len(_sfho_T) - 1))
_n_extra   = int(np.ceil(np.log(0.1 / 0.01) / np.log(_log_ratio)))   # ≈ 24
_T_extra   = np.geomspace(0.01, 0.1, _n_extra, endpoint=False)       # 0.01 ≤ T < 0.1

# SFHO log grid in [0.10, 1.0] MeV  (same points as Compose)
_T_low = _sfho_T[_sfho_T <= 1.0]
# Linear grid 2 → 150 MeV, step 2
_T_high = np.arange(2.0, 120.0 + 1e-9, 2.0)
T_values = np.concatenate([_T_extra, _T_low, _T_high])

# --- YC_vector: identical to SFHO ----------------------------------------------
Y_C_values = _sfho_grids.Y_C_grid

# Local/Global charge neutrality parameter η: 0 = GCN, 1 = LCN
eta_values = [0., 0.1, 0.3, 0.6, 1.0]

# vMIT bag model parameters
B4 = 165.0   # Bag constant B^{1/4} in MeV
a = 0.2      # Vector coupling in fm²
vmit_params = VMITParams(name=f"vMIT_B{int(B4)}_a{a}", B4=B4, a=a)

# ZL parameters
zl_params = ZLParams.default() #default for parameters used in Constantinou et. al 2023 and 2025.

# Output directories (absolute paths)
#from eos import REPO_ROOT
#DIR = str(REPO_ROOT / "output" / "zlvmit")
#os.makedirs(DIR, exist_ok=True)
DIR = os.path.abspath("../output/zlvmit_simulation_test")
os.makedirs(DIR, exist_ok=True)

# Control flags
VERBOSE = True                       # Print detailed progress
FORCE_RECOMPUTE_BOUNDARIES = True   # Recompute boundaries if they exist

# Print configuration summary
print("=" * 70)
print("ZL + vMIT Hybrid EOS Configuration")
print("=" * 70)
print(f"  Density grid: {len(n_B_values)} values from {n_B_values.min():.1f} to {n_B_values.max():.1f} fm^-3")
print(f"  Temperature grid: {len(T_values)} values from {T_values.min():.1f} to {T_values.max():.1f} MeV")
print(f"  η values: {eta_values}")
print(f"  vMIT parameters: B^(1/4) = {B4} MeV, a = {a} fm²")
print(f"  Output directory: {DIR}")
print("=" * 70)

In [ ]:
# =============================================================================
# INPUT PARAMETERS [alternative to the previous one]
# =============================================================================


# Density grid (fm^-3)
n0 = 0.16  # Nuclear saturation density
n_B_min = 0.1 * n0        
n_B_max = 12.0 * n0        
n_B_steps = 300           # Number of density points
n_B_values = np.linspace(n_B_min, n_B_max, n_B_steps)

# Temperature grid (MeV)
T_values=np.concatenate([[0, 0.1], np.arange(2.5, 121., 2.5)]) #np.logspace(-2,2.2,100) 

# Local/Global charge neutrality parameter η: 0 = GCN, 1 = LCN
eta_values = [0., 0.1, 0.3, 0.6, 1.0]

# Charge fractions (used in fixed YC case)
Y_C_values = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5] #np.arange(0.01, 0.50001, 0.01) 

# vMIT bag model parameters
B4 = 165.0   # Bag constant B^{1/4} in MeV
a = 0.2      # Vector coupling in fm²
vmit_params = VMITParams(name=f"vMIT_B{int(B4)}_a{a}", B4=B4, a=a)

# ZL parameters
zl_params = ZLParams.default() #default for parameters used in Constantinou et. al 2023 and 2025.

# Output directories (absolute paths)
#from eos import REPO_ROOT
#DIR = str(REPO_ROOT / "output" / "zlvmit")
#os.makedirs(DIR, exist_ok=True)
DIR = os.path.abspath("../output/zlvmit")
os.makedirs(DIR, exist_ok=True)

# Control flags
VERBOSE = True                       # Print detailed progress
FORCE_RECOMPUTE_BOUNDARIES = True   # Recompute boundaries if they exist

# Print configuration summary
print("=" * 70)
print("ZL + vMIT Hybrid EOS Configuration")
print("=" * 70)
print(f"  Density grid: [{n_B_min:.4f}, {n_B_max:.4f}] fm⁻³ ({n_B_steps} points)")
print(f"  Temperature grid: {len(T_values)} values from {T_values.min():.1f} to {T_values.max():.1f} MeV")
print(f"  η values: {eta_values}")
print(f"  vMIT parameters: B^(1/4) = {B4} MeV, a = {a} fm²")
print(f"  Output directory: {DIR}")
print("=" * 70)


# BETA EQUILIBRIUM 


## COMPUTE PURE PHASE TABLES (ZL and vMIT)

In [ ]:
# =============================================================================
# STEP 1: COMPUTE PURE PHASE TABLES (ZL and vMIT)
# =============================================================================

print("\n" + "=" * 70)
print("STEP 1: Computing Pure Phase Tables")
print("=" * 70)

# ZL pure phase table settings
zl_settings = ZLTableSettings(
    params=zl_params,
    n_B_values=n_B_values,
    T_values=T_values,
    equilibrium='beta_eq',
    include_photons=True,
    print_results=VERBOSE,
    print_first_n=1
)

# vMIT pure phase table settings  
vmit_settings = VMITTableSettings(
    params=vmit_params,
    n_B_values=n_B_values,
    T_values=T_values,
    equilibrium='beta_eq',
    include_photons=True,
    print_results=VERBOSE,
    print_first_n=1
)

# Compute ZL table
print("Computing ZL pure phase table...")
zl_table_by_T = compute_zl_table(zl_settings)
print(f"ZL table: {len(zl_table_by_T)} temperature points computed")

# Compute vMIT table
print("Computing vMIT pure phase table...")
vmit_table_by_T = compute_vmit_table(vmit_settings)
print(f"vMIT table: {len(vmit_table_by_T)} temperature points computed")

# Convert _table_by_T[(T),results]->_table[(n_B, T),results] lookup format needed for boundary finder
zl_table = {}
for key, results in zl_table_by_T.items():
    T = key[0]  # key is (T,) for beta_eq
    for i, r in enumerate(results):
        if r.converged:
            zl_table[(n_B_values[i], T)] = r

vmit_table = {}
for key, results in vmit_table_by_T.items():
    T = key[0]  # key is (T,) for beta_eq
    for i, r in enumerate(results):
        if r.converged:
            vmit_table[(n_B_values[i], T)] = r

print(f"Converted: {len(zl_table)} ZL points, {len(vmit_table)} vMIT points")

# Save pure phase tables
print("\nSaving pure phase tables...")

# Save as .dat with selected columns

# Convert _table_by_T[(T),results]->_results[results] for table writing
zl_results = [r for results in zl_table_by_T.values() for r in results]
vmit_results = [r for results in vmit_table_by_T.values() for r in results]

# ZL columns
zl_dat = {'nB': [], 'T': [], 'converged': [], 'error': [], 'mu_p_H': [], 'mu_n_H': [], 'n_p_H': [], 'n_n_H': [], 
          'mu_eL_H': [], 'n_eL_H': [], 'P_total': [], 'e_total': [], 's_total': [], 'f_total': [], 
          'Y_p_H': [], 'Y_n_H': [], 'Y_C_H': [], 'Y_S_H': [], 'Y_eL_H': []}

for r in zl_results:
    zl_dat['nB'].append(r.n_B)
    zl_dat['T'].append(r.T)
    zl_dat['converged'].append(r.converged)
    zl_dat['error'].append(r.error)
    zl_dat['mu_p_H'].append(r.mu_p)
    zl_dat['mu_n_H'].append(r.mu_n)
    zl_dat['n_p_H'].append(r.n_p)
    zl_dat['n_n_H'].append(r.n_n)
    zl_dat['mu_eL_H'].append(r.mu_e)
    zl_dat['n_eL_H'].append(r.n_e)
    zl_dat['P_total'].append(r.P_total)
    zl_dat['e_total'].append(r.e_total)
    zl_dat['s_total'].append(r.s_total)
    zl_dat['f_total'].append(r.e_total - r.T * r.s_total)
    zl_dat['Y_p_H'].append(r.Y_p)
    zl_dat['Y_n_H'].append(r.Y_n)
    zl_dat['Y_C_H'].append(r.Y_p)
    zl_dat['Y_S_H'].append(0.0)
    zl_dat['Y_eL_H'].append(r.Y_e)

# vMIT columns
vmit_dat = {'nB': [], 'T': [], 'converged': [], 'error': [], 'mu_u_Q': [], 'mu_d_Q': [], 'mu_s_Q': [], 
            'n_u_Q': [], 'n_d_Q': [], 'n_s_Q': [], 'mu_eL_Q': [], 'n_eL_Q': [], 'P_total': [], 'e_total': [], 
            's_total': [], 'f_total': [], 'Y_u_Q': [], 'Y_d_Q': [], 'Y_s_Q': [], 'Y_C_Q': [], 'Y_S_Q': [], 'Y_eL_Q': []}

for r in vmit_results:
    vmit_dat['nB'].append(r.n_B)
    vmit_dat['T'].append(r.T)
    vmit_dat['converged'].append(r.converged)
    vmit_dat['error'].append(r.error)
    vmit_dat['mu_u_Q'].append(r.mu_u)
    vmit_dat['mu_d_Q'].append(r.mu_d)
    vmit_dat['mu_s_Q'].append(r.mu_s)
    vmit_dat['n_u_Q'].append(r.n_u)
    vmit_dat['n_d_Q'].append(r.n_d)
    vmit_dat['n_s_Q'].append(r.n_s)
    vmit_dat['mu_eL_Q'].append(r.mu_e)
    vmit_dat['n_eL_Q'].append(r.n_e)
    vmit_dat['P_total'].append(r.P_total)
    vmit_dat['e_total'].append(r.e_total)
    vmit_dat['s_total'].append(r.s_total)
    vmit_dat['f_total'].append(r.e_total - r.T * r.s_total)
    vmit_dat['Y_u_Q'].append(r.Y_u)
    vmit_dat['Y_d_Q'].append(r.Y_d)
    vmit_dat['Y_s_Q'].append(r.Y_s)
    vmit_dat['Y_C_Q'].append(2/3 * r.Y_u - 1/3 * r.Y_d - 1/3 * r.Y_s)
    vmit_dat['Y_S_Q'].append(r.Y_s)
    vmit_dat['Y_eL_Q'].append(r.Y_e)

save_table_full(zl_dat, os.path.join(DIR, f"pure_H_table_beta.dat"), f"ZL pure H, beta eq")
save_table_full(vmit_dat, os.path.join(DIR, f"pure_Q_table_beta_B{int(B4)}_a{a}.dat"), f"vMIT pure Q, beta eq, B^1/4={B4} MeV")


## COMPUTE PHASE BOUNDARIES FOR EACH η

In [ ]:

# =============================================================================
# STEP 2: COMPUTE PHASE BOUNDARIES FOR EACH η
# =============================================================================

print("\n" + "=" * 70)
print("STEP 2: Computing Phase Boundaries")
print("=" * 70)

boundaries = {}
for eta in eta_values:
    print(f"Computing boundaries for η = {eta}")
    boundaries[eta] = get_or_compute_boundaries(
        eta=eta,
        T_values=T_values,
        zl_params=zl_params,
        vmit_params=vmit_params,
        output_dir=DIR,
        force_recompute=FORCE_RECOMPUTE_BOUNDARIES,
        verbose=VERBOSE,
        H_table_lookup=zl_table,
        Q_table_lookup=vmit_table,
        return_dict=True  # Returns dict format ready for generate_unified_table
    )
    print(f"  Found {len(boundaries[eta])} boundary points")


# GENERATE UNIFIED HYBRID EOS TABLES and save

In [ ]:

# =============================================================================
# STEP 3: GENERATE UNIFIED HYBRID EOS TABLES
# =============================================================================

print("\n" + "=" * 70)
print("STEP 3: Generating Unified Hybrid EOS Tables")
print("=" * 70)

hybrid_tables = {}
for eta in eta_values:
    print(f"Generating hybrid table for η = {eta}")
    unified_table = generate_unified_table(
        n_B_values=n_B_values,
        T_values=T_values,
        eta=eta,
        zl_params=zl_params,
        vmit_params=vmit_params,
        H_table=zl_table,
        Q_table=vmit_table,
        boundaries=boundaries[eta],
        verbose=VERBOSE
    )
    hybrid_tables[eta] = unified_table
    print(f"  Generated table with {len(unified_table)} total points")

# =============================================================================
# STEP 4: SAVE RESULTS TO FILES
# =============================================================================

print("\n" + "=" * 70)
print("STEP 4: Saving Results")
print("=" * 70)

for eta, table in hybrid_tables.items():
    filename_base = f"table_hybrid_betaeq_eta{eta:.2f}_B{int(B4)}_a{a}"
    header = f"ZL+vMIT hybrid EOS, eta={eta}, B^1/4={B4} MeV, a={a} fm^2, beta_eq"
    save_table_full(results_to_dict_primary(table, eta_override=eta), os.path.join(DIR, f"{filename_base}_result.dat"), header)
    save_table_full(results_to_dict_complete(table, eta_override=eta), os.path.join(DIR, f"{filename_base}_complete.dat"), header)

print("\n" + "=" * 70)
print("HYBRID EOS COMPUTATION COMPLETE")
print(f"Results saved to: {DIR}")
print("=" * 70)




# Fixed YC

## COMPUTE ALL PURE PHASE TABLES FOR ALL Y_C VALUES

In [ ]:
# -------------------------------------------------------------------------
# STEP 1: COMPUTE ALL PURE PHASE TABLES FOR ALL Y_C VALUES
# -------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 1: Computing Pure Phase Tables for all Y_C values")
print("=" * 70)

# Storage for all pure phase tables
all_H_tables_yc = {}
all_Q_tables_yc = {}

# Storage for unified tables
zl_dat_all = {'nB': [], 'Y_C': [], 'T': [], 'converged': [], 'error': [], 'mu_p_H': [], 'mu_n_H': [], 
              'n_p_H': [], 'n_n_H': [], 'mu_eL_H': [], 'n_eL_H': [], 'P_total': [], 'e_total': [], 
              's_total': [], 'f_total': [], 'Y_p_H': [], 'Y_n_H': [], 'Y_C_H': [], 'Y_S_H': [], 'Y_eL_H': []}

vmit_dat_all = {'nB': [], 'Y_C': [], 'T': [], 'converged': [], 'error': [], 'mu_u_Q': [], 'mu_d_Q': [], 'mu_s_Q': [], 
                'n_u_Q': [], 'n_d_Q': [], 'n_s_Q': [], 'mu_eL_Q': [], 'n_eL_Q': [], 'P_total': [], 'e_total': [], 
                's_total': [], 'f_total': [], 'Y_u_Q': [], 'Y_d_Q': [], 'Y_s_Q': [], 'Y_C_Q': [], 'Y_S_Q': [], 'Y_eL_Q': []}


# ZL pure phase table settings
zl_settings = ZLTableSettings(
    params=zl_params,
    n_B_values=n_B_values,
    T_values=T_values,
    equilibrium='fixed_yc',
    Y_C_values=Y_C_values,
    include_photons=True,
    print_results=VERBOSE,
    print_first_n=1
    )
    
# vMIT pure phase table settings
vmit_settings = VMITTableSettings(
    params=vmit_params,
    n_B_values=n_B_values,
    T_values=T_values,
    equilibrium='fixed_yc',
    Y_C_values=Y_C_values,
    include_photons=True,
    print_results=VERBOSE,
    print_first_n=1
    )
    
# Compute ZL table
print(f"Computing ZL pure phase")
zl_table_by_YC_T = compute_zl_table(zl_settings)

# Compute vMIT table
print(f"Computing vMIT pure phase")
vmit_table_by_YC_T = compute_vmit_table(vmit_settings)

    
# Convert _table_by_YC_T[(T,Y_C),results] --> zl_table_yc[Y_C][(n_B, T),results] lookup format for boundaries computation
zl_table_yc = {}
 
for key, results in zl_table_by_YC_T.items():
    T = key[0]  
    Y_C = key[1]  # Extract Y_C from the key
    
    # Initialize nested dict for this Y_C if not exists
    if Y_C not in zl_table_yc:
        zl_table_yc[Y_C] = {}
    
    for i, r in enumerate(results):
        zl_table_yc[Y_C][(n_B_values[i], T)] = r
 
vmit_table_yc = {}
for key, results in vmit_table_by_YC_T.items():
    T = key[0]  
    Y_C = key[1]  # Extract Y_C from the key
    
    # Initialize nested dict for this Y_C if not exists
    if Y_C not in vmit_table_yc:
        vmit_table_yc[Y_C] = {}
    
    for i, r in enumerate(results):
        vmit_table_yc[Y_C][(n_B_values[i], T)] = r




# Save pure phase tables
print("\nSaving pure phase tables...")

# Convert  _table_by_YC_T[(T,Y_C),results] --> _results[results] for table writing
zl_results = [r for results in zl_table_by_YC_T.values() for r in results]
vmit_results = [r for results in vmit_table_by_YC_T.values() for r in results]

# ZL columns
zl_dat = {'nB': [], 'Y_C': [], 'T': [], 'converged': [], 'error': [], 'mu_p_H': [], 'mu_n_H': [], 'n_p_H': [], 'n_n_H': [], 
          'mu_eL_H': [], 'n_eL_H': [], 'P_total': [], 'e_total': [], 's_total': [], 'f_total': [], 
          'Y_p_H': [], 'Y_n_H': [], 'Y_C_H': [], 'Y_S_H': [], 'Y_eL_H': []}

for r in zl_results:
    zl_dat['nB'].append(r.n_B)
    zl_dat['Y_C'].append(r.Y_C)
    zl_dat['T'].append(r.T)
    zl_dat['converged'].append(r.converged)
    zl_dat['error'].append(r.error)
    zl_dat['mu_p_H'].append(r.mu_p)
    zl_dat['mu_n_H'].append(r.mu_n)
    zl_dat['n_p_H'].append(r.n_p)
    zl_dat['n_n_H'].append(r.n_n)
    zl_dat['mu_eL_H'].append(r.mu_e)
    zl_dat['n_eL_H'].append(r.n_e)
    zl_dat['P_total'].append(r.P_total)
    zl_dat['e_total'].append(r.e_total)
    zl_dat['s_total'].append(r.s_total)
    zl_dat['f_total'].append(r.e_total - r.T * r.s_total)
    zl_dat['Y_p_H'].append(r.Y_p)
    zl_dat['Y_n_H'].append(r.Y_n)
    zl_dat['Y_C_H'].append(r.Y_p)
    zl_dat['Y_S_H'].append(0.0)
    zl_dat['Y_eL_H'].append(r.Y_e)

# vMIT columns
vmit_dat = {'nB': [],'Y_C': [], 'T': [], 'converged': [], 'error': [], 'mu_u_Q': [], 'mu_d_Q': [], 'mu_s_Q': [], 
            'n_u_Q': [], 'n_d_Q': [], 'n_s_Q': [], 'mu_eL_Q': [], 'n_eL_Q': [], 'P_total': [], 'e_total': [], 
            's_total': [], 'f_total': [], 'Y_u_Q': [], 'Y_d_Q': [], 'Y_s_Q': [], 'Y_C_Q': [], 'Y_S_Q': [], 'Y_eL_Q': []}

for r in vmit_results:
    vmit_dat['nB'].append(r.n_B)
    vmit_dat['Y_C'].append(r.Y_C)
    vmit_dat['T'].append(r.T)
    vmit_dat['converged'].append(r.converged)
    vmit_dat['error'].append(r.error)
    vmit_dat['mu_u_Q'].append(r.mu_u)
    vmit_dat['mu_d_Q'].append(r.mu_d)
    vmit_dat['mu_s_Q'].append(r.mu_s)
    vmit_dat['n_u_Q'].append(r.n_u)
    vmit_dat['n_d_Q'].append(r.n_d)
    vmit_dat['n_s_Q'].append(r.n_s)
    vmit_dat['mu_eL_Q'].append(r.mu_e)
    vmit_dat['n_eL_Q'].append(r.n_e)
    vmit_dat['P_total'].append(r.P_total)
    vmit_dat['e_total'].append(r.e_total)
    vmit_dat['s_total'].append(r.s_total)
    vmit_dat['f_total'].append(r.e_total - r.T * r.s_total)
    vmit_dat['Y_u_Q'].append(r.Y_u)
    vmit_dat['Y_d_Q'].append(r.Y_d)
    vmit_dat['Y_s_Q'].append(r.Y_s)
    vmit_dat['Y_C_Q'].append(2/3 * r.Y_u - 1/3 * r.Y_d - 1/3 * r.Y_s)
    vmit_dat['Y_S_Q'].append(r.Y_s)
    vmit_dat['Y_eL_Q'].append(r.Y_e)

save_table_full(zl_dat, os.path.join(DIR, f"pure_H_table_fixedYC.dat"), f"ZL pure H, beta eq")
save_table_full(vmit_dat, os.path.join(DIR, f"pure_Q_table_fixedYC_B{int(B4)}_a{a}.dat"), f"vMIT pure Q, beta eq, B^1/4={B4} MeV")

## COMPUTE PHASE BOUNDARIES FOR EACH η, Y_C

In [ ]:
# =============================================================================
# STEP 2: COMPUTE PHASE BOUNDARIES FOR EACH η, Y_C
# =============================================================================

print("\n" + "=" * 70)
print("STEP 2: Computing Phase Boundaries")
print("=" * 70)

boundaries = {}
for Y_C in Y_C_values:
    print("\n" + "=" * 70)
    print("Computing Phase Boundaries for Y_C = ", Y_C)
    print("=" * 70) 
    boundaries_YC = {}
    for eta in eta_values:
        print(f"Computing boundaries for η = {eta}, Y_C = {Y_C}")
        boundaries_YC[eta] = get_or_compute_boundaries(
        eta=eta,
        eq_mode="fixed_yc", 
        Y_C=Y_C,
        T_values=T_values,
        zl_params=zl_params,
        vmit_params=vmit_params,
        output_dir=DIR,
        force_recompute=FORCE_RECOMPUTE_BOUNDARIES,
        verbose=True,
        H_table_lookup=zl_table_yc[Y_C],
        Q_table_lookup=vmit_table_yc[Y_C],
        return_dict=True 
    )
    boundaries[Y_C]=boundaries_YC


## GENERATE UNIFIED HYBRID EOS TABLES FOR ALL Y_C AND η and save

In [ ]:
# -------------------------------------------------------------------------
# STEP 3: GENERATE UNIFIED HYBRID EOS TABLES FOR ALL Y_C AND η
# -------------------------------------------------------------------------
 
print("\n" + "=" * 70)
print("STEP 3: Generating unified hybrid EOS tables for all Y_C and η")
print("=" * 70)
 
all_hybrid_tables = {}
 
for Y_C in Y_C_values:
    print(f"\nGenerating tables for Y_C = {Y_C}...")
    
    hybrid_tables_yc = {}
    
    for eta in eta_values:
            
        print(f"  η = {eta:.2f}: ", end="", flush=True)
        
        hybrid_tables_yc[eta] = generate_unified_table(
            n_B_values=n_B_values,
            T_values=T_values,
            eta=eta,
            zl_params=zl_params,
            vmit_params=vmit_params,
            H_table=zl_table_yc[Y_C],      
            Q_table=vmit_table_yc[Y_C],    
            boundaries=boundaries[Y_C][eta],
            verbose=VERBOSE,
            eq_mode='fixed_yc',
            Y_C=Y_C
        )
        
    
    all_hybrid_tables[Y_C] = hybrid_tables_yc
    
# Save tables: one file per eta (containing all Y_C)
print("\n" + "-" * 70)
print("Saving tables (one per η)...")
 
for eta in eta_values:
    all_eta_results = []
    for Y_C in Y_C_values:
        if eta in all_hybrid_tables[Y_C]:
            all_eta_results.extend(all_hybrid_tables[Y_C][eta])

 
    filename_base = f"table_hybrid_fixedYC_eta{eta:.2f}_B{int(B4)}_a{a}"
    header = f"ZL+vMIT hybrid EOS, eta={eta}, fixed Y_C mode, B^1/4={B4} MeV, a={a} fm^2"
 
    save_table_full(results_to_dict_primary(all_eta_results, eta_override=eta),
                    os.path.join(DIR, f"{filename_base}_results.dat"), header)
    save_table_full(results_to_dict_complete(all_eta_results, eta_override=eta),
                    os.path.join(DIR, f"{filename_base}_complete.dat"), header)


print("\n" + "=" * 70)
print(f"Results saved to: {DIR}")
print("=" * 70)

# Plots 

In [ ]:
from eos.zlvmit.table_reader import EOSCollection

# ==============================================================================
# LOAD EOS TABLES
# ==============================================================================

base_path = DIR#REPO_ROOT / "output" / "zlvmit_eos"  #DIR

print("Loading EOS tables...")
eos_betaeq = EOSCollection(base_path, eq_mode='betaeq',
                           eta_values=eta_values,
                           B4=B4, a=a,                  
                           verbose=True)
eos_fixedyc = EOSCollection(base_path, eq_mode='fixedyc',
                            eta_values=eta_values,
                            Y_C_values=Y_C_values,
                            B4=B4, a=a,                 
                            verbose=True)



# ==============================================================================
# HOW TO USE EOSCollection
# ==============================================================================
#
# The .get() method interpolates any quantity at arbitrary (nB, T, eta):
#
#   value = eos_betaeq.get(quantity, nB, T=T, eta=eta)
#   value = eos_fixedyc.get(quantity, nB, T=T, eta=eta, Y_C=Y_C)
#
# AVAILABLE QUANTITIES:
# ---------------------
#   'chi'       - Quark volume fraction (0=hadron, 1=quark)
#   'P_total'   - Total pressure [MeV/fm³]
#   'e_total'   - Total energy density [MeV/fm³]
#   'f_total'   - Total free energy density [MeV/fm³]
#   's_total'   - Total entropy density [1/fm³]
#   'Y_p_tot'   - Proton fraction
#   'Y_n_tot'   - Neutron fraction
#   'Y_u_tot'   - Up quark fraction
#   'Y_d_tot'   - Down quark fraction
#   'Y_s_tot'   - Strange quark fraction
#   'Y_e_tot'   - Electron fraction
#   'Y_C_H'     - Charge fraction in hadronic phase
#   'Y_C_Q'     - Charge fraction in quark phase
#   ... and more (see table headers)
#
# EXAMPLES:
# ---------
#   # Get a single value: pressure at nB=0.5 fm⁻³, T=50 MeV, η=0.3
#   P = eos_betaeq.get('P_total', 0.5, T=50, eta=0.3)
#
#   # Get values along a density array (list comprehension)
#   nB_arr = np.linspace(0.1, 1.5, 100)
#   P_arr = np.array([eos_betaeq.get('P_total', nB, T=50, eta=0.3) for nB in nB_arr])
#   chi_arr = np.array([eos_betaeq.get('chi', nB, T=0, eta=0.0) for nB in nB_arr])
#
#   # Multiple quantities at once
#   nB_arr = np.linspace(0.1, 1.5, 100)
#   P_arr = np.array([eos_betaeq.get('P_total', nB, T=0, eta=0.0) for nB in nB_arr])
#   e_arr = np.array([eos_betaeq.get('e_total', nB, T=0, eta=0.0) for nB in nB_arr])
#
#   # For fixed Y_C mode, specify Y_C
#   P_arr = np.array([eos_fixedyc.get('P_total', nB, T=50, eta=0.3, Y_C=0.5) for nB in nB_arr])
#
# PHASE BOUNDARIES:
# -----------------
#   # Get onset/offset arrays for phase diagram
#   T_arr, nB_onset, nB_offset = eos_betaeq.get_boundary_arrays(eta=0.3)
#   T_arr, nB_onset, nB_offset = eos_fixedyc.get_boundary_arrays(eta=0.3, Y_C=0.5)
#
# ==============================================================================


In [ ]:
# ==============================================================================
# Full audit of loaded EOS tables:
#   - per-file general info (path, size, grid extents, columns)
#   - convergence summary (failed points + locations)
#   - large-error points (converged but error above threshold)
# ==============================================================================
import os

ERR_THRESHOLD = 1e-4   # rows with error > this are flagged "large error"
TOP_K         = 10     # how many worst-error rows to show per table

def _audit(name, reader, fixedyc):
    d  = reader.data
    fp = reader.filename
    nB = d['n_B']; T = d['T']; conv = d['converged']; err = d['error']
    YC = d.get('Y_C', None) if fixedyc else None

    # --- General info ---
    size_mb = os.path.getsize(fp) / 1e6 if os.path.exists(fp) else float('nan')
    nB_u    = np.unique(nB)
    T_u     = np.unique(T)
    YC_u    = np.unique(YC) if YC is not None else None
    print(f"  [{name}]  file: {os.path.basename(fp)}  ({size_mb:.1f} MB, {len(nB)} rows, {len(reader.data)} cols)")
    print(f"     n_B grid: {len(nB_u):4d} pts  range [{nB_u.min():.4f}, {nB_u.max():.4f}] fm^-3")
    print(f"     T   grid: {len(T_u):4d} pts  range [{T_u.min():.4f}, {T_u.max():.4f}] MeV")
    if YC is not None:
        print(f"     Y_C grid: {len(YC_u):4d} pts  range [{YC_u.min():.3f}, {YC_u.max():.3f}]")

    # --- Convergence ---
    bad      = conv < 0.5
    n_bad    = int(bad.sum())
    big_err  = (~bad) & (err > ERR_THRESHOLD)
    n_big    = int(big_err.sum())
    status   = "OK  " if (n_bad == 0 and n_big == 0) else "WARN"
    print(f"     [{status}]  converged: {len(bad) - n_bad}/{len(bad)}   "
          f"large-error (err > {ERR_THRESHOLD:g}): {n_big}   "
          f"max err: {err.max():.2e}")

    # --- Failed points (first TOP_K) ---
    if n_bad:
        idx = np.flatnonzero(bad)[:TOP_K]
        print(f"     ✗ first {len(idx)} non-converged rows  (n_B, T{', Y_C' if fixedyc else ''}, error):")
        for i in idx:
            extra = f", Y_C={YC[i]:.3f}" if fixedyc else ""
            print(f"        n_B={nB[i]:7.4f}  T={T[i]:7.3f}{extra}  err={err[i]:.2e}")

    # --- Top-K largest-error converged rows ---
    if n_big:
        # rank by error descending, only among converged-but-noisy rows
        idx_sorted = np.flatnonzero(big_err)
        idx_sorted = idx_sorted[np.argsort(-err[idx_sorted])][:TOP_K]
        print(f"     ⚠ top {len(idx_sorted)} converged-but-noisy rows by error:")
        for i in idx_sorted:
            extra = f", Y_C={YC[i]:.3f}" if fixedyc else ""
            print(f"        n_B={nB[i]:7.4f}  T={T[i]:7.3f}{extra}  err={err[i]:.2e}")

    print()


print("=" * 78)
print("EOS audit: eos_betaeq")
print("=" * 78)
for eta, r in sorted(eos_betaeq.eos_tables.items()):
    _audit(f"eta={eta}", r, fixedyc=False)

print("=" * 78)
print("EOS audit: eos_fixedyc")
print("=" * 78)
for eta, r in sorted(eos_fixedyc.eos_tables.items()):
    _audit(f"eta={eta}", r, fixedyc=True)


In [ ]:
from eos.zlvmit.plot_results import (
    # Main plotting functions
    plot_betaeq,
    plot_fixedyc,
    plot_composition,
    plot_mixed_phase_boundaries,
    # Style setup
    setup_matplotlib_style,
    # Constants (if needed for custom plots)
    N_SAT,
    ETA_STYLES,
    SPECIES_STYLES,
    SPECIES_GROUPS,
)

# Configure matplotlib 
setup_matplotlib_style()


## chi(nB,T)

In [ ]:
# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
YC_T_set = [(0.01, 10), (0.5, 10), (0.01, 100), (0.5, 100)]

for (i, j), (Y_C, T) in zip(positions, YC_T_set):
    plot_fixedyc(axes[i, j], eos_fixedyc, 'chi', n_B_values, T, Y_C, eta_values, xlim=(0.1, 12), ylim=(-0.05, 1.05))

plt.tight_layout()
plt.show()






# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
T_plot = [0.01,20,50,120]

for (i, j), T in zip(positions, T_plot):
    plot_betaeq(axes[i, j], eos_betaeq, 'chi', n_B_values, T, eta_values, xlim=(0.1, 12), ylim=(-0.05, 1.05))

plt.tight_layout()
plt.show()

plt.tight_layout()

## Boundaries

In [ ]:
# ==============================================================================
# T(nB) PHASE DIAGRAM - Complete self-contained version
# ==============================================================================


# Create figure
fig, axes = plt.subplots(2, 2, figsize=(15, 15))

# Beta equilibrium
plot_mixed_phase_boundaries(axes[0, 0], eos_betaeq, eta_values, Y_C=None,
                            normalize_nB=True, xlim=(.1, 12), ylim=(0, 120))
axes[0, 0].set_title(r'$\beta$-equilibrium')

# Fixed Y_C panels
for (i, j), YC in zip([(0, 1), (1, 0), (1, 1)], [0.1, 0.3, 0.5]):
    plot_mixed_phase_boundaries(axes[i, j], eos_fixedyc, eta_values, Y_C=YC,
                                normalize_nB=True, xlim=(.1, 12), ylim=(0, 120))
    axes[i, j].set_title(rf'Fixed $Y_C = {YC}$')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# T(μ_B) PHASE DIAGRAM - direct read from phase_boundaries_*.dat
# ==============================================================================
# Reads μ_B (= mu_n_H at onset; equals the coexistence baryon chemical
# potential by Gibbs equilibrium) directly from the saved boundary files.

import os
import numpy as np

def _load_boundary_T_muB(eta, eq_mode, B4, a, Y_C=None, base_dir=DIR):
    suffix = 'betaeq' if eq_mode == 'betaeq' else 'fixedYC'
    fname = f"phase_boundaries_eta{eta:.2f}_{suffix}_B{int(B4)}_a{a}.dat"
    data = np.loadtxt(os.path.join(base_dir, fname))
    if eq_mode == 'betaeq':
        # Columns: T n_B_onset n_B_offset conv mu_p_H mu_n_H ...
        T = data[:, 0]
        mu_B = data[:, 5]
    else:
        # Columns: Y_C T n_B_onset n_B_offset conv mu_p_H mu_n_H ...
        if Y_C is not None:
            data = data[np.abs(data[:, 0] - Y_C) < 1e-3]
        T = data[:, 1]
        mu_B = data[:, 6]
    return T, mu_B


fig, axes = plt.subplots(2, 2, figsize=(15, 15))

# Beta equilibrium
ax = axes[0, 0]
for eta in eta_values:
    T, mu_B = _load_boundary_T_muB(eta, 'betaeq', B4, a)
    style = ETA_STYLES.get(eta, {'color': 'black', 'linestyle': '-', 'linewidth': 4.0})
    ax.plot(mu_B, T,
            color=style['color'], linestyle=style['linestyle'],
            linewidth=style['linewidth'], label=style.get('label', f'η={eta}'))
ax.set_xlabel(r'$\mu_B$ [MeV]')
ax.set_ylabel(r'$T$ [MeV]')
ax.set_title(r'$\beta$-equilibrium')
ax.legend(frameon=False, loc='best')
ax.grid(True, alpha=0.3, linestyle=':')

# Fixed Y_C panels
for (i, j), YC in zip([(0, 1), (1, 0), (1, 1)], [0.1, 0.3, 0.5]):
    ax = axes[i, j]
    for eta in eta_values:
        T, mu_B = _load_boundary_T_muB(eta, 'fixedyc', B4, a, Y_C=YC)
        style = ETA_STYLES.get(eta, {'color': 'black', 'linestyle': '-', 'linewidth': 4.0})
        ax.plot(mu_B, T,
                color=style['color'], linestyle=style['linestyle'],
                linewidth=style['linewidth'], label=style.get('label', f'η={eta}'))
    ax.set_xlabel(r'$\mu_B$ [MeV]')
    ax.set_ylabel(r'$T$ [MeV]')
    ax.set_title(rf'Fixed $Y_C = {YC}$')
    ax.legend(frameon=False, loc='best')
    ax.grid(True, alpha=0.3, linestyle=':')

plt.tight_layout()
plt.show()


## Pressure

In [ ]:
# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
YC_T_set = [(0.1, 10), (0.4, 10), (0.1, 50), (0.4, 50)]

for (i, j), (Y_C, T) in zip(positions, YC_T_set):
    plot_fixedyc(axes[i, j], eos_fixedyc, 'P_total', n_B_values, T, Y_C, eta_values, xlim=(0.1, 12), ylim=(0, 1500))

plt.tight_layout()
plt.show()






# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
T_plot = [0.1,20,70,120]

for (i, j), T in zip(positions, T_plot):
    plot_betaeq(axes[i, j], eos_betaeq, 'P_total', n_B_values, T, eta_values, xlim=(0.1, 12), ylim=(0, 1500))

plt.tight_layout()
plt.show()

plt.tight_layout()

## Energy, Entropy, and Free Energy

In [ ]:
# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
YC_T_set = [(0.1, 10), (0.4, 10), (0.1, 50), (0.4, 50)]

for (i, j), (Y_C, T) in zip(positions, YC_T_set):
    plot_fixedyc(axes[i, j], eos_fixedyc, 'F_specific', n_B_values, T, Y_C, eta_values, xlim=(0.1, 12), ylim=(500, 1800))

plt.tight_layout()
plt.show()






# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
T_plot = [0,20,50,70]

for (i, j), T in zip(positions, T_plot):
    plot_betaeq(axes[i, j], eos_betaeq, 'F_specific', n_B_values, T, eta_values, xlim=(0.1, 12), ylim=(500, 1800))

plt.tight_layout()
plt.show()

plt.tight_layout()

In [ ]:
# Define figure size
figsize = (16, 16)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
YC_T_set = [(0.1, 10), (0.4, 10), (0.1, 50), (0.4, 50)]

for (i, j), (Y_C, T) in zip(positions, YC_T_set):
    plot_fixedyc(axes[i, j], eos_fixedyc, 'S_specific', n_B_values, T, Y_C, eta_values, xlim=(0.1, 12), ylim=(0, 4))

plt.tight_layout()
plt.show()






# Define figure size
figsize = (16, 16)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
T_plot = [0.1,20,50,70]

for (i, j), T in zip(positions, T_plot):
    plot_betaeq(axes[i, j], eos_betaeq, 'S_specific', n_B_values, T, eta_values, xlim=(0.1, 12), ylim=(0, 5))

plt.tight_layout()
plt.show()

plt.tight_layout()

In [ ]:
# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
YC_T_set = [(0.1, 10), (0.4, 10), (0.1, 50), (0.4, 50)]

for (i, j), (Y_C, T) in zip(positions, YC_T_set):
    plot_fixedyc(axes[i, j], eos_fixedyc, 'e_per_nB', n_B_values, T, Y_C, eta_values, xlim=(0.1, 12), ylim=(800, 1800))

plt.tight_layout()
plt.show()






# Define figure size
figsize = (15, 15)

fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
T_plot = [0,20,50,70]

for (i, j), T in zip(positions, T_plot):
    plot_betaeq(axes[i, j], eos_betaeq, 'e_per_nB', n_B_values, T, eta_values, xlim=(0.1, 12), ylim=(800, 1800))

plt.tight_layout()
plt.show()

plt.tight_layout()

In [ ]:
from eos.zlvmit.plot_results import _get_quantity_value, ETA_STYLES, QUANTITY_INFO, N_SAT

def plot_fixedyc_vsT(ax, eos_collection, quantity, T_values, nB, YC, eta_values,
                     xlim=None, ylim=None):
    for eta in eta_values:
        style = ETA_STYLES.get(eta, {'color': 'black', 'linestyle': '-', 'linewidth': 4.0})
        values = [_get_quantity_value(eos_collection, quantity, nB, T=T, eta=eta, Y_C=YC)
                  for T in T_values]
        ax.plot(T_values, values,
                color=style['color'], linestyle=style['linestyle'],
                linewidth=style['linewidth'], label=style.get('label', f'η={eta}'))

    q_info = QUANTITY_INFO.get(quantity, {'label': quantity, 'unit': ''})
    ax.set_xlabel(r'$T$ [MeV]')
    ax.set_ylabel(q_info['label'] + (f" {q_info['unit']}" if q_info['unit'] else ""))
    ax.set_title(rf'$n_B/n_0={nB/N_SAT:.2f}$, $Y_C={YC}$')
    ax.legend(frameon=False, loc='best')
    ax.grid(True, alpha=0.3, linestyle=':')
    if xlim is not None: ax.set_xlim(xlim)
    if ylim is not None: ax.set_ylim(ylim)


figsize = (15, 15)
fig, axes = plt.subplots(2, 2, figsize=figsize)

positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
nB_YC_set = [(0.5*N_SAT, 0.01), (0.5*N_SAT, 0.5),
             (3.0*N_SAT, 0.01), (3.0*N_SAT, 0.5)]

for (i, j), (nB, Y_C) in zip(positions, nB_YC_set):
    plot_fixedyc_vsT(axes[i, j], eos_fixedyc, 'chi', T_values, nB, Y_C, eta_values,
                     xlim=(T_values.min(), T_values.max()), ylim=(-0.05, 1.05))

plt.tight_layout()
plt.show()


## Composition

In [ ]:
## Composition Y_i(nB) - Beta equilibrium

# Plot basic composition (Y_p, Y_n, Y_u, Y_d, Y_s, Y_e)
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
eta_T_set = [(0.0, 10), (0.0, 100), (1, 10), (1, 100)]

for (i, j), (eta, T) in zip(positions, eta_T_set):
    plot_composition(axes[i, j], eos_betaeq, n_B_values, T, eta, 
                     species='basic', xlim=(0.1, 12), ylim=(0, 1.2))

plt.tight_layout()
plt.show()

## Composition Y_i(nB) - Fixed YC

fig, axes = plt.subplots(3, 2, figsize=(15, 20))
positions = [(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1)]
eta_YC_T_set = [(0.0, 0.1, 10), (0.0, 0.4, 10), (0.6, 0.1, 50), (0.6, 0.4, 50), (1, 0.1, 10), (1, 0.4, 50)]

for (i, j), (eta, YC, T) in zip(positions, eta_YC_T_set):
    plot_composition(axes[i, j], eos_fixedyc, n_B_values, T, eta, 
                     species='basic', Y_C=YC, xlim=(0.1, 12), ylim=(0, 1.2))

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Y_i(T) at fixed n_B and η  (composition vs temperature, beta-equilibrium)
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (nB [fm^-3], eta) per panel
nB_eta_set = [(0.1*n0, 0.0), (n0, 0.0), (0.1*n0, 1.0), (n0, 1.0)]
positions  = [(0, 0), (0, 1), (1, 0), (1, 1)]

T_arr = np.array([t for t in T_values if t > 0])  # skip T=0

species_list = ['Y_p_tot', 'Y_n_tot', 'Y_u_tot', 'Y_d_tot', 'Y_s_tot', 'Y_e_tot']

for (i, j), (nB, eta) in zip(positions, nB_eta_set):
    ax = axes[i, j]
    for sp in species_list:
        Y = np.array([eos_betaeq.get(sp, nB, T=T, eta=eta) for T in T_arr])
        style = SPECIES_STYLES[sp]
        ax.plot(T_arr, Y,
                color=style['color'], linestyle=style['linestyle'],
                linewidth=style['linewidth'], label=style['label'])
    ax.set_xlabel(r'$T$ [MeV]')
    ax.set_ylabel(r'$Y_i$')
    ax.set_title(rf'$n_B = {nB:.2f}\ \mathrm{{fm}}^{{-3}}$, $\eta = {eta}$')
    ax.set_ylim(0, 1.2)
    ax.legend(frameon=False, loc='best', ncol=2)
    ax.grid(True, alpha=0.3, linestyle=':')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Full-grid crust attach with ADAPTIVE δn per (η, T, Y_C)
# Each combo widens δn until P_crust(n_tr - δn) ≤ P_core(n_tr + δn), then blends.
# Residual monotone-clamp on entries that still dip (rare, only at the cap).
# ==============================================================================
import time
from scipy.interpolate import PchipInterpolator
from scipy.optimize   import brentq
from eos.astro.tov.solver   import EOSTable_for_TOV, add_crust, load_crust_table

T_grid_attach  = T_values
YC_grid_attach = Y_C_values

n_transition = 0.3 * n0          # blend center
delta_n_init = 0.2 * n0          # starting band half-width
DELTA_N_MAX  = 0.5 * n0          # widening cap (~0.08 fm^-3)

# ---- tqdm fallback --------------------------------------------------------
try:
    from tqdm.notebook import tqdm
except ImportError:
    class _NoBar:
        def __init__(self, total, **k): self.n = 0; self.total = total
        def update(self, k=1):
            self.n += k
            if self.n % max(1, self.total // 20) == 0:
                print(f"  {self.n}/{self.total}")
        def close(self): pass
    tqdm = lambda total, **k: _NoBar(total, **k)

# ---- helper: smallest δn s.t. P_crust(n_tr-δn) ≤ P_core(n_tr+δn) ---------
def _adaptive_delta_n(P_crust_fn, P_core_fn, n_tr, dn_init, dn_max):
    def F(dn):
        return float(P_crust_fn(n_tr - dn) - P_core_fn(n_tr + dn))
    if F(dn_init) <= 0:
        return dn_init
    if F(dn_max) > 0:
        return dn_max
    return float(brentq(F, dn_init, dn_max))

# ---- pre-warm SFHO 3-D load ----------------------------------------------
_warm = EOSTable_for_TOV(P=np.array([0.1, 100.0]),
                         epsilon=np.array([10.0, 1000.0]),
                         nB=np.array([0.05, 1.0]))
add_crust(_warm, 'compose_sfho_nYCT', 'interpolate',
          n_transition=n_transition, delta_n=delta_n_init,
          crust_T=float(T_grid_attach[0]), crust_Y_C=float(YC_grid_attach[0]),
          verbose=False)

# ---- main loop ------------------------------------------------------------
nB_arr   = np.asarray(n_B_values)
n_combos = len(eta_values) * len(T_grid_attach) * len(YC_grid_attach)
print(f"Building {n_combos} attached EOS tables "
      f"({len(eta_values)} η × {len(T_grid_attach)} T × {len(YC_grid_attach)} Y_C)...")

attached_eos = {}
skipped_oob  = 0
skipped_err  = 0
n_widened    = 0
n_capped     = 0
n_clamped    = 0

t0  = time.time()
bar = tqdm(total=n_combos, desc="attach")
for eta in eta_values:
    for T in T_grid_attach:
        for YC in YC_grid_attach:
            # 1. core slice (vectorized)
            P_core = eos_fixedyc.get('P_total', nB_arr, T=T, eta=eta, Y_C=YC)
            e_core = eos_fixedyc.get('e_total', nB_arr, T=T, eta=eta, Y_C=YC)

            mask = np.isfinite(P_core) & np.isfinite(e_core)
            if mask.sum() < 2:
                skipped_oob += 1
                bar.update(1); continue

            P_core_m = P_core[mask]
            e_core_m = e_core[mask]
            nB_m     = nB_arr[mask]

            # 2. crust load + δn root-find
            try:
                crust = load_crust_table('compose_sfho_nYCT', T=T, Y_C=YC)
                P_crust_fn = PchipInterpolator(crust.nB, crust.P, extrapolate=True)
                P_core_fn  = PchipInterpolator(nB_m,    P_core_m, extrapolate=True)
                dn_use = _adaptive_delta_n(P_crust_fn, P_core_fn,
                                            n_transition, delta_n_init, DELTA_N_MAX)
            except Exception:
                # crust-side problem — fall back to default δn, will likely still dip
                dn_use = delta_n_init

            if dn_use > delta_n_init + 1e-12:
                n_widened += 1
            if dn_use >= DELTA_N_MAX - 1e-12:
                n_capped += 1

            # 3. rebuild merged with that δn
            core = EOSTable_for_TOV(P=P_core_m, epsilon=e_core_m, nB=nB_m)
            try:
                merged = add_crust(core, 'compose_sfho_nYCT', 'interpolate',
                                   n_transition=n_transition, delta_n=dn_use,
                                   crust_T=float(T), crust_Y_C=float(YC),
                                   verbose=False)
            except ValueError:
                skipped_err += 1
                bar.update(1); continue

            # 4. last-resort monotone clamp if still dipping
            if np.any(np.diff(merged.P) < 0):
                P_fix   = np.maximum.accumulate(merged.P)
                muB     = (merged.P + merged.epsilon) / merged.nB
                eps_fix = muB * merged.nB - P_fix
                merged  = EOSTable_for_TOV(P=P_fix, epsilon=eps_fix, nB=merged.nB)
                n_clamped += 1

            attached_eos[(float(eta), float(T), float(YC))] = merged
            bar.update(1)
bar.close()
dt = time.time() - t0

print(f"\n✓ {len(attached_eos)}/{n_combos} attached tables built in {dt:.1f} s")
print(f"  skipped_oob (core all NaN) = {skipped_oob}")
print(f"  skipped_err (blend failed) = {skipped_err}")
print(f"  widened δn beyond init     = {n_widened}")
print(f"  hit DELTA_N_MAX cap        = {n_capped}")
print(f"  needed residual clamp      = {n_clamped}")
print(f"  sum (built + skipped)      = {len(attached_eos) + skipped_oob + skipped_err}  "
      f"(should equal {n_combos})")


# ==============================================================================
# Audit
# ==============================================================================
TOL = 0.0
TOP_K = 20

bad = []
for (eta, T, YC), m in attached_eos.items():
    if len(m.P) < 2: continue
    dP = np.diff(m.P)
    if np.any(dP < TOL):
        i_neg = np.where(dP < TOL)[0]
        worst_step = int(i_neg[np.argmin(dP[i_neg])])
        bad.append({
            'eta': eta, 'T': T, 'YC': YC,
            'n_decreasing_steps': len(i_neg),
            'worst_drop':         float(dP[worst_step]),
            'worst_drop_frac':    float(dP[worst_step]) / max(float(m.P[worst_step]), 1e-30),
            'worst_nB':           float(m.nB[worst_step]),
            'nB_region':          (float(m.nB[i_neg.min()]), float(m.nB[i_neg.max() + 1])),
        })

print(f"\nAudit: {len(attached_eos)} merged EOSs scanned")
print(f"  ✓ monotone:     {len(attached_eos) - len(bad)}")
print(f"  ✗ non-monotone: {len(bad)}")

if bad:
    bad.sort(key=lambda b: b['worst_drop'])
    print(f"\nTop {min(TOP_K, len(bad))} worst offenders:")
    print(f"  {'eta':>5s} {'T':>8s} {'Y_C':>6s} {'n_bad':>5s}  "
          f"{'worst_ΔP':>10s} {'frac':>8s}  worst_n_B   region (n_B)")
    for b in bad[:TOP_K]:
        print(f"  {b['eta']:5.2f} {b['T']:8.3f} {b['YC']:6.3f} {b['n_decreasing_steps']:5d}  "
              f"{b['worst_drop']:10.3e} {b['worst_drop_frac']:8.1%}  "
              f"{b['worst_nB']:.4f}    "
              f"[{b['nB_region'][0]:.4f}, {b['nB_region'][1]:.4f}]")


In [ ]:
# ==============================================================================
# At fixed (η, T, Y_C):  P vs n_B,  P vs ε,  P vs μ_B   in LIN-LIN
#   • SFHO crust (Compose) ............... dotted
#   • ZL+vMIT core only .................. dashed
#   • Unified (blended) .................. solid
#
# μ_B uses the Gibbs relation with FREE energy density:  μ_B = (P + f) / n_B
# (correct at finite T;  reduces to (P+ε)/n_B only at T=0)
# ==============================================================================
from eos.astro.tov.solver import load_crust_table, EOSTable_for_TOV, add_crust
from eos.general.compose import ComposeLookup
from scipy.interpolate import PchipInterpolator

# --- pick what to compare ---
eta_plot = 1
T_plot   = 0.1     # MeV  (must be inside your saved EOS T grid)
YC_plot  = 0.5

n0       = 0.16    # nuclear saturation density [fm^-3]

# Optional zoom (linear axes; tighter limits help see the crust+blend region)
xlim_nB  = (0.0, 0.16)
xlim_eps = (0.0, 500.0)
ylim_P   = (0.0, 4.0)



# --- ZL+vMIT core slice (vectorized) ---
nB_arr  = np.asarray(n_B_values)
P_core  = eos_fixedyc.get('P_total', nB_arr, T=T_plot, eta=eta_plot, Y_C=YC_plot)
e_core  = eos_fixedyc.get('e_total', nB_arr, T=T_plot, eta=eta_plot, Y_C=YC_plot)
f_core  = eos_fixedyc.get('f_total', nB_arr, T=T_plot, eta=eta_plot, Y_C=YC_plot)
muB_core = (P_core + f_core) / nB_arr        # Gibbs μ = (P+f)/n_B

# --- SFHO crust slice (use lookup, not load_crust_table — we need f too) ---
_sfho = ComposeLookup('/Users/mircoguerrini/Desktop/Research/Compose/SFHO_Compose',
                         name='sfho_for_plot')
sf = _sfho.get_slice(T_plot, YC_plot)
crust_nB, crust_P, crust_e, crust_f = sf['n_B'], sf['P'], sf['epsilon'], sf['f']
muB_crust = (crust_P + crust_f) / crust_nB

# --- Unified (use attached_eos if available; else build on the fly) ---
key = (eta_plot, float(T_plot), float(YC_plot))
if 'attached_eos' in globals() and key in attached_eos:
    merged = attached_eos[key]
else:
    _core = EOSTable_for_TOV(P=P_core, epsilon=e_core, nB=nB_arr)
    merged = add_crust(_core, 'compose_sfho_nYCT', 'interpolate',
                       n_transition=n_transition, delta_n=delta_n,
                       crust_T=T_plot, crust_Y_C=YC_plot, verbose=False)

# --- reconstruct f for merged via the SAME tanh blend the library uses for P ---
f_crust_interp = PchipInterpolator(crust_nB, crust_f, extrapolate=True)
f_core_interp  = PchipInterpolator(nB_arr,   f_core,  extrapolate=True)

def _blend_w(n, n_tr, dn):
    return 0.5 * (1.0 + np.tanh((n - n_tr) / (dn / 2.0)))

w = _blend_w(merged.nB, n_transition, delta_n)
f_merged = np.where(
    merged.nB <= n_transition - delta_n, f_crust_interp(merged.nB),
    np.where(
        merged.nB >= n_transition + delta_n, f_core_interp(merged.nB),
        (1 - w) * f_crust_interp(merged.nB) + w * f_core_interp(merged.nB),
    ),
)
muB_merged = (merged.P + f_merged) / merged.nB

# --- x-limit for the μ_B panel: μ_B(n_B=0.1·n0) → μ_B(n_B=1.0·n0) ---
muB_of_nB = PchipInterpolator(merged.nB, muB_merged, extrapolate=False)
xlim_muB  = sorted([float(muB_of_nB(0.1 * n0)), float(muB_of_nB(1.0 * n0))])

# ==============================================================================
# Plot
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(22, 6))

# Panel 1: P vs n_B
ax = axes[0]
ax.plot(crust_nB,  crust_P,   ls=':',  lw=2.0, color='C0', label='SFHO crust (Compose)')
ax.plot(nB_arr,    P_core,    ls='--', lw=2.0, color='C3', label='ZL+vMIT core')
ax.plot(merged.nB, merged.P,  ls='-',  lw=2.0, color='k',  label='Unified (blended)')
ax.axvspan(n_transition - delta_n, n_transition + delta_n, color='gray', alpha=0.15,
           label=r'blend band $n_\mathrm{tr} \pm \delta_n$')
ax.set_xlabel(r'$n_B$ [fm$^{-3}$]')
ax.set_ylabel(r'$P$ [MeV/fm$^3$]')
ax.set_xlim(*xlim_nB); ax.set_ylim(*ylim_P)
ax.set_title(rf'$\eta={eta_plot}$, $T={T_plot:g}$ MeV, $Y_C={YC_plot}$')
ax.legend(frameon=False, loc='best'); ax.grid(True, alpha=0.3)

# Panel 2: P vs ε
ax = axes[1]
ax.plot(crust_e,  crust_P,   ls=':',  lw=2.0, color='C0', label='SFHO crust (Compose)')
ax.plot(e_core,   P_core,    ls='--', lw=2.0, color='C3', label='ZL+vMIT core')
ax.plot(merged.epsilon, merged.P, ls='-', lw=2.0, color='k', label='Unified (blended)')
ax.set_xlabel(r'$\varepsilon$ [MeV/fm$^3$]')
ax.set_ylabel(r'$P$ [MeV/fm$^3$]')
ax.set_xlim(*xlim_eps); ax.set_ylim(*ylim_P)
ax.legend(frameon=False, loc='best'); ax.grid(True, alpha=0.3)

# Panel 3: P vs μ_B = (P + f) / n_B
#ax = axes[2]
#ax.plot(muB_crust,  crust_P,   ls=':',  lw=2.0, color='C0', label='SFHO crust (Compose)')
#ax.plot(muB_core,   P_core,    ls='--', lw=2.0, color='C3', label='ZL+vMIT core')
#ax.plot(muB_merged, merged.P,  ls='-',  lw=2.0, color='k',  label='Unified (blended)')
#ax.axvline(float(muB_of_nB(0.1 * n0)), color='gray', alpha=0.4, lw=1, ls='--')
#ax.axvline(float(muB_of_nB(1.0 * n0)), color='gray', alpha=0.4, lw=1, ls='--')
#ax.set_xlabel(r'$\mu_B$ [MeV]')
#ax.set_ylabel(r'$P$ [MeV/fm$^3$]')
#ax.set_xlim(*xlim_muB); ax.set_ylim(*ylim_P)
#ax.legend(frameon=False, loc='best'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# --- diagnostic print ---
print(f"μ_B(crust) range [n_B>0.01]: [{muB_crust[crust_nB > 1e-2].min():.0f}, {muB_crust[crust_nB > 1e-2].max():.0f}] MeV")
print(f"μ_B(core)  range           : [{muB_core.min():.0f}, {muB_core.max():.0f}] MeV")
print(f"μ_B(merged) range          : [{muB_merged.min():.0f}, {muB_merged.max():.0f}] MeV")
print(f"x-limits used:               [{xlim_muB[0]:.1f}, {xlim_muB[1]:.1f}] MeV"
      f"  (μ_B at n_B=0.1·n0 and 1·n0)")


Generating final tables

In [ ]:
# ==============================================================================
# Full-eta export of ZL+vMIT fixedYC tables (wsub + nosub, .dat + .h5)
# 5 etas × {wsub, nosub} × {dat, h5}  +  README.txt  →  20 data files + 1 readme
# Output folder: <DIR>/ZLvMIT_tables/
#
# _wsub uses an extended n_B grid: Compose n_B (below n_B_values[0]) followed
# by the ZLvMIT n_B_values. Crust blend = same adaptive-δn as the attach cell.
# Three regions per (T, Y_C) slice:
#   chi = -1.0     subnuclear (pure SFHO Compose)
#   chi = -0.5     interpolation band (linear blend of EVERY column)
#   chi ∈ [0, 1]   pure ZL+vMIT core
# Thermo: P, e, s, μ, Y all linearly blended with the same δn; f := e − sT
# is enforced everywhere (analytic in subnuclear/core; explicit in band and
# after the monotone P-clamp).
# ==============================================================================
# Requires h5py:  pip install h5py
import os, time
from datetime import datetime
import numpy as np
import h5py
from scipy.interpolate import PchipInterpolator
from scipy.optimize   import brentq

# ---- Sanity ------------------------------------------------------------------
if 'eos_fixedyc' not in globals():
    raise NameError("Run the EOSCollection load cell first (defines eos_fixedyc).")
if '_sfho_grids' not in globals():
    raise NameError("Run the ComposeLookup cell first (defines _sfho_grids).")

# ---- Knobs -------------------------------------------------------------------
OUT_DIR      = os.path.join(DIR, 'ZLvMIT_tables')
os.makedirs(OUT_DIR, exist_ok=True)

N_TRANS      = 0.3 * n0
DELTA_N_INIT = 0.2 * n0
DELTA_N_MAX  = 0.5 * n0
CHI_SUB_SENTINEL  = -1.0
CHI_BAND_SENTINEL = -0.5

# ---- Columns: (output_name, source_in_eos_fixedyc_or_None, units) ------------
COLUMNS = [
    ('n_B',     None,      'fm^-3'),
    ('T',       None,      'MeV'),
    ('Y_C',     None,      '-'),
    ('P',       'P_total', 'MeV/fm^3'),
    ('e',       'e_total', 'MeV/fm^3'),
    ('f',       'f_total', 'MeV/fm^3'),
    ('s',       's_total', '1/fm^3'),
    ('mu_p',    'mu_p_H',  'MeV'),
    ('mu_n',    'mu_n_H',  'MeV'),
    ('mu_u',    'mu_u_Q',  'MeV'),
    ('mu_d',    'mu_d_Q',  'MeV'),
    ('mu_s',    'mu_s_Q',  'MeV'),
    ('mu_B',    None,      'MeV'),
    ('mu_C',    None,      'MeV'),
    ('mu_S',    None,      'MeV'),
    ('mu_eH',   'mu_eL_H', 'MeV'),
    ('mu_eQ',   'mu_eL_Q', 'MeV'),
    ('mu_eG',   'mu_eG',   'MeV'),
    ('Y_p_tot', 'Y_p_tot', '-'),
    ('Y_n_tot', 'Y_n_tot', '-'),
    ('Y_u_tot', 'Y_u_tot', '-'),
    ('Y_d_tot', 'Y_d_tot', '-'),
    ('Y_s_tot', 'Y_s_tot', '-'),
    ('Y_S_tot', 'Y_S_tot', '-'),
    ('Y_e_tot', 'Y_e_tot', '-'),
    ('chi',     'chi',     '-'),
]
COL_NAMES  = [c[0] for c in COLUMNS]
COL_SOURCE = {c[0]: c[1] for c in COLUMNS}
COL_UNITS  = {c[0]: c[2] for c in COLUMNS}

# ---- Adaptive-δn helpers (mirror the attach cell) ----------------------------
def _adaptive_dn(P_crust_fn, P_core_fn, n_tr, dn_init, dn_max):
    F = lambda dn: float(P_crust_fn(n_tr - dn) - P_core_fn(n_tr + dn))
    if F(dn_init) <= 0: return dn_init
    if F(dn_max)  > 0:  return dn_max
    return float(brentq(F, dn_init, dn_max))

def _interp(target, x, y, extrapolate=False):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2:
        return np.full_like(target, np.nan, dtype=float)
    return PchipInterpolator(x[m], y[m], extrapolate=extrapolate)(target)

def _blend_band_safe(core, crust, nB, n_tr, dn):
    """Pure crust below n_tr-dn, pure core above n_tr, linear blend in band.
    NaN core in the band falls back to crust (used in the extended subnuclear
    grid where ZLvMIT has no data)."""
    pure_crust = nB <= (n_tr - dn)
    in_band    = (~pure_crust) & (nB < n_tr)
    w          = np.clip((nB - (n_tr - dn)) / dn, 0.0, 1.0)
    core_safe  = np.where(np.isfinite(core), core, crust)
    out = core.copy()
    out = np.where(pure_crust, crust, out)
    out = np.where(in_band, (1.0 - w) * crust + w * core_safe, out)
    return out

# ---- Grids -------------------------------------------------------------------
nB_arr  = np.asarray(n_B_values)
YC_arr  = np.asarray(Y_C_values)
T_arr_g = np.asarray(T_values)
n_nB, n_YC, n_T = len(nB_arr), len(YC_arr), len(T_arr_g)
shape3d = (n_nB, n_YC, n_T)

# Extended nB grid for _wsub: Compose nB points below 0.016 + ZLvMIT n_B_values
_sfho_nB_full   = np.asarray(_sfho_grids.n_B)
_extra_nB_below = _sfho_nB_full[_sfho_nB_full < nB_arr[0]]
nB_arr_wsub     = np.concatenate([_extra_nB_below, nB_arr])
n_extra         = len(_extra_nB_below)
print(f"_wsub extended grid: +{n_extra} Compose points below {nB_arr[0]:.4f} fm^-3 "
      f"(lowest n_B = {nB_arr_wsub[0]:.2e} fm^-3)")

# ---- Cache core arrays per eta (fetch each column once) ----------------------
_core_cache = {}
def get_core_3d(ETA):
    if ETA in _core_cache:
        return _core_cache[ETA]
    direct = [(c, s) for c, s in COL_SOURCE.items() if s is not None]
    print(f"  fetching {len(direct)} core columns × {n_YC*n_T} slices ...")
    t0 = time.time()
    cache = {}
    for col_out, col_src in direct:
        arr3d = np.full(shape3d, np.nan)
        for j, YC in enumerate(YC_arr):
            for k, T in enumerate(T_arr_g):
                arr3d[:, j, k] = eos_fixedyc.get(col_src, nB_arr,
                                                 T=T, eta=ETA, Y_C=YC)
        cache[col_out] = arr3d
    chi = cache['chi']
    cache['mu_B'] = (1.0 - chi) * cache['mu_n'] + chi * (2.0*cache['mu_d'] + cache['mu_u'])
    cache['mu_C'] = (1.0 - chi) * (cache['mu_p'] - cache['mu_n']) + chi * (cache['mu_u'] - cache['mu_d'])
    cache['mu_S'] = chi * (cache['mu_s'] - cache['mu_d'])    # μ_S_H = 0
    print(f"  core fetch in {time.time()-t0:.1f} s")
    _core_cache[ETA] = cache
    return cache

# ---- Build one table for (ETA, WITH_CRUST) -----------------------------------
def build_table(ETA, WITH_CRUST):
    core = get_core_3d(ETA)

    if not WITH_CRUST:
        nB_used = nB_arr
        A = {c: core[c].copy() for c in core if c in COL_NAMES}
        # Add chi (already in core), then coordinate columns
        nBg, YCg, Tg = np.meshgrid(nB_used, YC_arr, T_arr_g, indexing='ij')
        A['n_B'] = nBg; A['T'] = Tg; A['Y_C'] = YCg
        return A, nB_used, {'widened':0,'capped':0,'clamped':0,'skipped':0,
                            'total':n_YC*n_T}

    # WITH_CRUST: extended grid
    nB_used = nB_arr_wsub
    shape_w = (len(nB_used), n_YC, n_T)
    A = {c: np.full(shape_w, np.nan) for c in COL_NAMES}
    nBg, YCg, Tg = np.meshgrid(nB_used, YC_arr, T_arr_g, indexing='ij')
    A['n_B'] = nBg; A['T'] = Tg; A['Y_C'] = YCg

    # Embed cached core into the upper part (indices ≥ n_extra)
    for c in COL_NAMES:
        if c in ('n_B', 'T', 'Y_C'): continue
        if c in core:
            A[c][n_extra:, :, :] = core[c]

    stats = {'widened':0,'capped':0,'clamped':0,'skipped':0,'total':n_YC*n_T}

    for j, YC_val in enumerate(YC_arr):
        for k, T_val in enumerate(T_arr_g):
            P_c = A['P'][:, j, k]
            e_c = A['e'][:, j, k]
            valid = np.isfinite(P_c) & np.isfinite(e_c)
            if valid.sum() < 2:
                stats['skipped'] += 1
                continue

            sf = _sfho_grids.get_slice(float(T_val), float(YC_val))
            m_sf = np.isfinite(sf['P']) & np.isfinite(sf['n_B'])
            if m_sf.sum() < 2:
                continue

            # 1. adaptive δn
            P_crust_fn = PchipInterpolator(sf['n_B'][m_sf], sf['P'][m_sf], extrapolate=True)
            P_core_fn  = PchipInterpolator(nB_used[valid],  P_c[valid],    extrapolate=True)
            try:
                dn_use = _adaptive_dn(P_crust_fn, P_core_fn,
                                      N_TRANS, DELTA_N_INIT, DELTA_N_MAX)
            except Exception:
                dn_use = DELTA_N_INIT
            if dn_use > DELTA_N_INIT + 1e-12: stats['widened'] += 1
            if dn_use >= DELTA_N_MAX  - 1e-12: stats['capped']  += 1

            # 2. SFHO/Compose values on the extended grid
            P_sf   = _interp(nB_used, sf['n_B'], sf['P'])
            e_sf   = _interp(nB_used, sf['n_B'], sf['epsilon'])
            s_sf   = _interp(nB_used, sf['n_B'], sf['s']) * nB_used
            muB_sf = _interp(nB_used, sf['n_B'], sf['mu_B'])
            muC_sf = _interp(nB_used, sf['n_B'], sf['mu_C'])
            muL_sf = _interp(nB_used, sf['n_B'], sf['mu_L'])

            # 3. SFHO-side values for every blended column
            zeros  = np.zeros_like(nB_used)
            YC_lin = np.full_like(nB_used, float(YC_val))   # Y_e = Y_C exact
            sfho_side = {
                'P':       P_sf,
                'e':       e_sf,
                's':       s_sf,
                'mu_B':    muB_sf,
                'mu_C':    muC_sf,
                'mu_S':    zeros,
                'mu_n':    muB_sf,
                'mu_p':    muB_sf + muC_sf,
                'mu_u':    zeros, 'mu_d': zeros, 'mu_s': zeros,
                'mu_eH':   muL_sf, 'mu_eQ': muL_sf, 'mu_eG': muL_sf,
                'Y_p_tot': zeros, 'Y_n_tot': zeros,
                'Y_u_tot': zeros, 'Y_d_tot': zeros, 'Y_s_tot': zeros,
                'Y_S_tot': zeros,
                'Y_e_tot': YC_lin,
            }

            # 4. NaN-safe linear blend for every column (Option D)
            for c, crust_arr in sfho_side.items():
                A[c][:, j, k] = _blend_band_safe(A[c][:, j, k], crust_arr,
                                                  nB_used, N_TRANS, dn_use)

            # 5. f := e − sT (thermo consistency by construction)
            A['f'][:, j, k] = A['e'][:, j, k] - A['s'][:, j, k] * float(T_val)

            # 6. Monotone clamp on P (sorted by nB); preserve μ_B for e; re-enforce f
            P_b = A['P'][:, j, k]
            e_b = A['e'][:, j, k]
            s_b = A['s'][:, j, k]
            order = np.argsort(nB_used)
            inv   = np.argsort(order)
            nB_s  = nB_used[order]
            P_s   = P_b[order]; e_s = e_b[order]
            good  = np.isfinite(P_s) & np.isfinite(e_s) & (nB_s > 0)
            if good.sum() >= 2:
                P_g = P_s[good]; e_g = e_s[good]; nB_g = nB_s[good]
                if np.any(np.diff(P_g) < 0):
                    muB_g = (P_g + e_g) / nB_g
                    P_fix = np.maximum.accumulate(P_g)
                    e_fix = muB_g * nB_g - P_fix
                    P_s[good] = P_fix
                    e_s[good] = e_fix
                    stats['clamped'] += 1
            A['P'][:, j, k] = P_s[inv]
            A['e'][:, j, k] = e_s[inv]
            A['f'][:, j, k] = A['e'][:, j, k] - s_b * float(T_val)

            # 7. chi sentinels: −1 subnuclear, −0.5 band, core chi elsewhere
            subnuclear = (nB_used < nB_arr[0]) | (nB_used <= N_TRANS - dn_use)
            in_band    = (~subnuclear) & (nB_used < N_TRANS)
            chi_b = A['chi'][:, j, k].copy()
            A['chi'][:, j, k] = np.where(subnuclear, CHI_SUB_SENTINEL,
                                np.where(in_band,    CHI_BAND_SENTINEL, chi_b))

    return A, nB_used, stats

# ---- Consistency report ------------------------------------------------------
def consistency_report(A, nB_used, label):
    T_b3d = np.broadcast_to(T_arr_g[None, None, :], A['P'].shape)
    resid = A['f'] - (A['e'] - A['s'] * T_b3d)
    fscl  = max(np.nanmax(np.abs(A['f'])), 1e-30)
    print(f"  [{label}] f-(e-sT): max|res|={np.nanmax(np.abs(resid)):.3e}, "
          f"max|res|/max|f|={np.nanmax(np.abs(resid))/fscl:.3e}, "
          f"med|res|={np.nanmedian(np.abs(resid)):.3e}")
    n_bad = 0; worst = 0.0; loc = None; checked = 0
    for j in range(A['P'].shape[1]):
        for k in range(A['P'].shape[2]):
            col = A['P'][:, j, k]
            m   = np.isfinite(col)
            if m.sum() < 2: continue
            checked += 1
            cf = col[m]; nf = nB_used[m]
            dP = np.diff(cf)
            if np.any(dP < 0):
                n_bad += 1
                iw = int(np.argmin(dP))
                if dP[iw] < worst:
                    worst = float(dP[iw])
                    loc   = (float(nf[iw]), float(T_arr_g[k]), float(YC_arr[j]))
    extra = (f"; worst at n_B={loc[0]:.4e}, T={loc[1]:.4g}, Y_C={loc[2]:.3f}"
             if loc else "")
    print(f"  [{label}] dP/dn_B>=0: {n_bad}/{checked} slices non-monotone, "
          f"worst ΔP={worst:.3e}{extra}")

# ---- Writers -----------------------------------------------------------------
def write_ascii(A, nB_used, path, ETA, with_crust):
    # Row order: Y_C outermost, T middle, n_B innermost
    flat = {c: A[c].transpose(1, 2, 0).ravel() for c in COL_NAMES}
    mode = ((f"SFHO crust blended, adaptive δn "
             f"(n_trans={N_TRANS:.4f}, δn_init={DELTA_N_INIT:.4f}, "
             f"δn_max={DELTA_N_MAX:.4f}); chi=-1 subnuclear, chi=-0.5 band, "
             "all μ and Y linearly blended in the band")
            if with_crust else "ZLvMIT only (no crust)")
    header = (
        "ZL+vMIT fixedYC\n"
        "Particles: n, p, u, d, s, e\n"
        f"eta = {ETA}, vMIT: B^(1/4) = {B4} MeV, a = {a} fm^2, mode: {mode}\n"
        f"Grid: n_nB = {len(nB_used)}, n_Y_C = {n_YC}, n_T = {n_T}\n"
        "Columns: " + "  ".join(f"{c}[{COL_UNITS[c]}]" for c in COL_NAMES)
    )
    save_table_full(flat, path, header)

def write_hdf5(A, nB_used, path, ETA, with_crust):
    with h5py.File(path, 'w') as h5:
        h5.attrs['description'] = "ZL+vMIT hybrid EOS fixedYC export"
        h5.attrs['eta']         = float(ETA)
        h5.attrs['B4_MeV']      = float(B4)
        h5.attrs['a_fm2']       = float(a)
        h5.attrs['equilibrium'] = "fixed_yc"
        h5.attrs['axis_order']  = "(n_B, Y_C, T)"
        h5.attrs['crust']       = "SFHO_blended_adaptive" if with_crust else "none"
        if with_crust:
            h5.attrs['n_transition']             = float(N_TRANS)
            h5.attrs['delta_n_init']             = float(DELTA_N_INIT)
            h5.attrs['delta_n_max']              = float(DELTA_N_MAX)
            h5.attrs['chi_subnuclear_sentinel']  = float(CHI_SUB_SENTINEL)
            h5.attrs['chi_band_sentinel']        = float(CHI_BAND_SENTINEL)
        for name, arr, units in [('n_B', nB_used, 'fm^-3'),
                                 ('Y_C', YC_arr,  '-'),
                                 ('T',   T_arr_g, 'MeV')]:
            d = h5.create_dataset(name, data=arr); d.attrs['units'] = units
        for c in COL_NAMES:
            if c in ('n_B', 'T', 'Y_C'): continue
            d = h5.create_dataset(c, data=A[c], compression='gzip')
            d.attrs['units'] = COL_UNITS[c]

def write_readme(path):
    lines = [
        "ZL+vMIT hybrid EOS — fixed-Y_C tables",
        f"Generated: {datetime.now().isoformat(timespec='seconds')}",
        "",
        f"ZLvMIT grid:  n_nB = {n_nB},  n_Y_C = {n_YC},  n_T = {n_T}",
        f"  n_B  ∈ [{nB_arr.min():.4f}, {nB_arr.max():.4f}] fm^-3",
        f"  Y_C  ∈ [{YC_arr.min():.3f}, {YC_arr.max():.3f}]",
        f"  T    ∈ [{T_arr_g.min():.3f}, {T_arr_g.max():.3f}] MeV",
        f"_wsub extended n_B grid: +{n_extra} Compose points below "
        f"{nB_arr[0]:.4f} fm^-3 (lowest n_B = {nB_arr_wsub[0]:.2e} fm^-3)",
        f"Eta values: {list(eta_values)}",
        f"vMIT:  B^(1/4) = {B4} MeV,  a = {a} fm^2",
        "",
        "ASCII (.dat) row order:  Y_C outermost, T middle, n_B innermost",
        "  i.e. all n_B at fixed (Y_C, T), then advance T, then advance Y_C.",
        "",
        "Particles: n, p, u, d, s, e",
        "",
        "Files per eta:",
        "  ZLvMIT_nBTYC_eta{eta:.2f}_wsub.{dat,h5}   — with SFHO Compose subnuclear",
        "  ZLvMIT_nBTYC_eta{eta:.2f}_nosub.{dat,h5}  — ZLvMIT core only",
        "",
        "Crust blend (only _wsub files):",
        f"  n_transition = {N_TRANS:.4f} fm^-3  (= 0.3·n0)",
        f"  δn_init      = {DELTA_N_INIT:.4f} fm^-3  (= 0.2·n0)",
        f"  δn_max       = {DELTA_N_MAX:.4f} fm^-3  (= 0.5·n0)",
        "  Adaptive δn per (T, Y_C): widen until P_crust(n_tr-δn) ≤ P_core(n_tr+δn).",
        "  Same δn used for ALL columns (P, e, s, μ, Y).",
        "  Monotone clamp on P with μ_B-preserving e adjustment.",
        "  f := e − sT enforced after the blend AND after the clamp.",
        "",
        "chi sentinels (only _wsub files):",
        "  chi = −1.0   subnuclear (pure SFHO Compose):",
        "    μ_n = μ_B,  μ_p = μ_B + μ_C,  μ_eH = μ_eQ = μ_eG = μ_L (all Compose),",
        "    μ_S = 0,  quark μ's = 0,  Y_u = Y_d = Y_s = Y_S = 0,",
        "    Y_e = Y_C (exact: charge neutrality),",
        "    Y_p, Y_n = 0 (Compose composition not parsed — would need eos.compo).",
        "  chi = −0.5   interpolation band:",
        "    EVERY column is linearly blended between the SFHO-side values above",
        "    and the ZLvMIT core values, with weight w = (n_B − (n_tr−δn))/δn.",
        "    Thermo identity f = e − sT holds by construction. Other identities",
        "    (Maxwell relations, sound speed from derivatives, Euler) are only",
        "    approximate across the seam.",
        "  chi ∈ [0, 1]  pure ZL+vMIT (everything physical, from the solver).",
        "",
        "Removed columns (would be constant across the table):",
        "  Y_B_tot = 1   (baryon-fraction normalization)",
        "  Y_C_tot = Y_C (slice value of the charge fraction)",
        "",
        "Derived chemical potentials in the core (S(s-quark) = +1 convention):",
        "  μ_B_H = μ_n,   μ_C_H = μ_p − μ_n,   μ_S_H = 0",
        "  μ_B_Q = 2·μ_d + μ_u,   μ_C_Q = μ_u − μ_d,   μ_S_Q = μ_s − μ_d",
        "  μ_B = (1−χ)·μ_B_H + χ·μ_B_Q   (same shape for μ_C, μ_S)",
        "",
        "Columns:",
    ]
    for c in COL_NAMES:
        lines.append(f"  {c:10s} [{COL_UNITS[c]}]")
    lines += [
        "",
        "HDF5 layout (per file):",
        "  /n_B   (n_nB,)             fm^-3",
        "  /Y_C   (n_YC,)             -",
        "  /T     (n_T,)              MeV",
        "  /<col> (n_nB, n_YC, n_T)   for each non-coordinate column",
        "  Root attrs: eta, B4_MeV, a_fm2, equilibrium, axis_order, crust,",
        "              n_transition, delta_n_init, delta_n_max,",
        "              chi_subnuclear_sentinel (-1.0),",
        "              chi_band_sentinel (-0.5)  — both in _wsub files only.",
    ]
    with open(path, 'w') as f:
        f.write("\n".join(lines) + "\n")
    print(f"  wrote {path}")

# ---- Main loop ---------------------------------------------------------------
t0_total = time.time()
print(f"Writing to {OUT_DIR}/")
for ETA in eta_values:
    for WITH_CRUST in (False, True):
        tag  = 'wsub' if WITH_CRUST else 'nosub'
        base = f"ZLvMIT_nBTYC_eta{ETA:.2f}_{tag}"
        print(f"\n[eta={ETA:.2f}, {tag}] building ...")
        t0 = time.time()
        A, nB_used, st = build_table(ETA, WITH_CRUST)
        msg = (f"slices ok={st['total']-st['skipped']}/{st['total']}"
               + (f", widened={st['widened']}, capped={st['capped']}, "
                  f"clamped={st['clamped']}" if WITH_CRUST else ""))
        print(f"  built in {time.time()-t0:.1f} s — {msg}  (n_nB={len(nB_used)})")
        consistency_report(A, nB_used, base)
        write_ascii(A, nB_used, os.path.join(OUT_DIR, f"{base}.dat"), ETA, WITH_CRUST)
        write_hdf5 (A, nB_used, os.path.join(OUT_DIR, f"{base}.h5"),  ETA, WITH_CRUST)

write_readme(os.path.join(OUT_DIR, "README.txt"))
print(f"\nAll done in {time.time()-t0_total:.1f} s.  Files in {OUT_DIR}")


In [ ]:
# ==============================================================================
# TEST: load exported HDF5 tables and plot P(n_B), P(e) for various (η, Y_C, T)
# Four figures × 4 panels each (lin-lin & log-log of both P vs n_B and P vs e)
# ==============================================================================
import os, h5py
import numpy as np
import matplotlib.pyplot as plt

TABLES_DIR = os.path.join(DIR, 'ZLvMIT_tables')

# ---- Load tables --------------------------------------------------------------
def _load(eta, tag):
    path = os.path.join(TABLES_DIR, f"ZLvMIT_nBTYC_eta{eta:.2f}_{tag}.h5")
    with h5py.File(path, 'r') as h5:
        return {k: h5[k][:] for k in ('n_B', 'Y_C', 'T', 'P', 'e', 'f', 's', 'chi')}

TABLES = {'wsub':  {eta: _load(eta, 'wsub')  for eta in eta_values},
          'nosub': {eta: _load(eta, 'nosub') for eta in eta_values}}

_e0 = eta_values[0]
print(f"Loaded {len(eta_values)} etas × 2 modes from {TABLES_DIR}")
print(f"  wsub  : n_nB = {len(TABLES['wsub'][_e0]['n_B']):4d}, "
      f"n_B ∈ [{TABLES['wsub'][_e0]['n_B'][0]:.2e}, {TABLES['wsub'][_e0]['n_B'][-1]:.3f}] fm^-3")
print(f"  nosub : n_nB = {len(TABLES['nosub'][_e0]['n_B']):4d}, "
      f"n_B ∈ [{TABLES['nosub'][_e0]['n_B'][0]:.4f}, {TABLES['nosub'][_e0]['n_B'][-1]:.3f}] fm^-3")
print(f"  Y_C   : {list(TABLES['wsub'][_e0]['Y_C'])}")
print(f"  T     : {len(TABLES['wsub'][_e0]['T'])} pts in "
      f"[{TABLES['wsub'][_e0]['T'][0]:.2f}, {TABLES['wsub'][_e0]['T'][-1]:.2f}] MeV")

# ---- Helpers ------------------------------------------------------------------
def _idx(grid, val):
    return int(np.argmin(np.abs(grid - val)))

def get_slice(eta, YC, T, mode='wsub'):
    """Return (n_B, P, e, chi) along n_B at given (eta, Y_C, T)."""
    tab = TABLES[mode][eta]
    iY = _idx(tab['Y_C'], YC)
    iT = _idx(tab['T'],   T)
    return (tab['n_B'], tab['P'][:, iY, iT],
            tab['e'][:, iY, iT], tab['chi'][:, iY, iT])

def make_4panel(title):
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes[0, 0].set(xlabel=r'$n_B$ [fm$^{-3}$]', ylabel=r'$P$ [MeV/fm$^3$]',
                   title='P vs $n_B$  (lin-lin)')
    axes[0, 1].set(xlabel=r'$e$ [MeV/fm$^3$]',  ylabel=r'$P$ [MeV/fm$^3$]',
                   title='P vs $e$  (lin-lin)')
    axes[1, 0].set(xlabel=r'$n_B$ [fm$^{-3}$]', ylabel=r'$P$ [MeV/fm$^3$]',
                   title='P vs $n_B$  (log-log)', xscale='log', yscale='log')
    axes[1, 1].set(xlabel=r'$e$ [MeV/fm$^3$]',  ylabel=r'$P$ [MeV/fm$^3$]',
                   title='P vs $e$  (log-log)',  xscale='log', yscale='log')
    fig.suptitle(title)
    return fig, axes

def plot_curve(axes, nB, P, e, label, color=None, ls='-'):
    kw = dict(label=label, lw=1.5, ls=ls)
    if color is not None: kw['color'] = color
    axes[0, 0].plot(nB, P, **kw)
    axes[0, 1].plot(e,  P, **kw)
    axes[1, 0].plot(nB, P, **kw)
    axes[1, 1].plot(e,  P, **kw)

def finalize(fig, axes):
    for ax in axes.flat:
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.show()

# ---- Knob ---------------------------------------------------------------------
MODE = 'wsub'      # 'wsub' or 'nosub' for figures 1–3

# ---- Figure 1: vary η at fixed (T, Y_C) --------------------------------------
T_fix, YC_fix = 10.0, 0.1
fig, axes = make_4panel(
    f"Vary η  (T = {T_fix} MeV,  Y_C = {YC_fix},  mode = {MODE})")
colors = plt.cm.viridis(np.linspace(0, 0.9, len(eta_values)))
for color, eta in zip(colors, eta_values):
    nB, P, e, _ = get_slice(eta, YC_fix, T_fix, MODE)
    plot_curve(axes, nB, P, e, f"η = {eta}", color)
finalize(fig, axes)

# ---- Figure 2: vary Y_C at fixed (η, T) --------------------------------------
eta_fix, T_fix = 0.0, 10.0
YC_list = list(TABLES[MODE][eta_fix]['Y_C'])
fig, axes = make_4panel(
    f"Vary $Y_C$  (η = {eta_fix},  T = {T_fix} MeV,  mode = {MODE})")
colors = plt.cm.plasma(np.linspace(0, 0.9, len(YC_list)))
for color, YC in zip(colors, YC_list):
    nB, P, e, _ = get_slice(eta_fix, YC, T_fix, MODE)
    plot_curve(axes, nB, P, e, f"$Y_C$ = {YC:.2f}", color)
finalize(fig, axes)

# ---- Figure 3: vary T at fixed (η, Y_C) --------------------------------------
eta_fix, YC_fix = 0.0, 0.1
T_choices = [0.01, 1.0, 10.0, 50.0, 100.0]
T_avail   = [TABLES[MODE][eta_fix]['T'][_idx(TABLES[MODE][eta_fix]['T'], t)]
             for t in T_choices]
fig, axes = make_4panel(
    f"Vary T  (η = {eta_fix},  Y_C = {YC_fix},  mode = {MODE})")
colors = plt.cm.inferno(np.linspace(0, 0.85, len(T_avail)))
for color, T_val in zip(colors, T_avail):
    nB, P, e, _ = get_slice(eta_fix, YC_fix, T_val, MODE)
    plot_curve(axes, nB, P, e, f"T = {T_val:.2g} MeV", color)
finalize(fig, axes)

# ---- Figure 4: wsub vs nosub overlay at one (η, Y_C, T) ----------------------
eta_fix, YC_fix, T_fix = 0.0, 0.1, 10.0
fig, axes = make_4panel(
    f"wsub vs nosub  (η = {eta_fix},  $Y_C$ = {YC_fix},  T = {T_fix} MeV)")
for tag, color, ls in (('wsub', 'C0', '-'), ('nosub', 'C3', '--')):
    nB, P, e, _ = get_slice(eta_fix, YC_fix, T_fix, tag)
    plot_curve(axes, nB, P, e, tag, color, ls)
finalize(fig, axes)


# TOV Equations

In [ ]:
### Compute the T=0 mixed phase hybrid EOS ZL+vMIT

# Directory for TOV outputs
from pathlib import Path
#from eos import REPO_ROOT
#OUTPUT_DIR_TOV = REPO_ROOT / "output" / "zlvmit" / "tov"
#OUTPUT_DIR_TOV.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_TOV = Path(DIR) / "tov"
OUTPUT_DIR_TOV.mkdir(parents=True, exist_ok=True)

## Pure phases

# ZL pure phase table settings
zl_settings = ZLTableSettings(
    params=zl_params,
    n_B_values=n_B_values,
    T_values=[0.],
    equilibrium='beta_eq',
    include_photons=True,
    print_results=VERBOSE,
    print_first_n=1
)

# vMIT pure phase table settings  
vmit_settings = VMITTableSettings(
    params=vmit_params,
    n_B_values=n_B_values,
    T_values=[0.],
    equilibrium='beta_eq',
    include_photons=True,
    print_results=VERBOSE,
    print_first_n=1
)

# Compute ZL table
zl_table_by_T = compute_zl_table(zl_settings)

# Compute vMIT table
vmit_table_by_T = compute_vmit_table(vmit_settings)


# Convert to lookup format
zl_table = {}
for key, results in zl_table_by_T.items():
    T = key[0]
    for i, r in enumerate(results):
        if r.converged:
            zl_table[(n_B_values[i], T)] = r

vmit_table = {}
for key, results in vmit_table_by_T.items():
    T = key[0]
    for i, r in enumerate(results):
        if r.converged:
            vmit_table[(n_B_values[i], T)] = r



# Compute boundaries
boundaries = {}
for eta in eta_values:
    print(f"Computing boundaries for η = {eta}")
    boundaries[eta] = get_or_compute_boundaries(
        eta=eta,
        T_values=[0.],
        zl_params=zl_params,
        vmit_params=vmit_params,
        output_dir=OUTPUT_DIR_TOV,  
        force_recompute=True,
        verbose=VERBOSE,
        H_table_lookup=zl_table,
        Q_table_lookup=vmit_table,
        return_dict=True
    )

# Generate hybrid tables
hybrid_tables_T0 = {}
for eta in eta_values:
    print(f"Generating hybrid table for η = {eta}")
    unified_table = generate_unified_table(
        n_B_values=n_B_values,
        T_values=[0.],
        eta=eta,
        zl_params=zl_params,
        vmit_params=vmit_params,
        H_table=zl_table,
        Q_table=vmit_table,
        boundaries=boundaries[eta],
        verbose=VERBOSE
    )
    hybrid_tables_T0[eta] = unified_table


# Export T=0 hybrid EOS to .dat 

for eta, results in hybrid_tables_T0.items():
    # Extract nB, P, epsilon from results
    nB_list, P_list, e_list = [], [], []
    
    for r in results:
        nB_list.append(r.n_B)
        P_list.append(r.P_total)
        e_list.append(r.e_total)
    

    # Convert to arrays and sort by nB
    nB = np.array(nB_list)
    P = np.array(P_list)
    epsilon = np.array(e_list)
    
    idx = np.argsort(nB)
    nB = nB[idx]
    P = P[idx]
    epsilon = epsilon[idx]
    
    # Write to file: columns = (P, epsilon, nB) to match default from_file
    filename = OUTPUT_DIR_TOV / f"eos_T0_betaeq_eta{eta:.2f}.dat"
    
    header = f"# ZL+vMIT hybrid EOS, T=0 MeV, beta-equilibrium, eta={eta:.2f}\n"
    header += f"# Columns: P [MeV/fm^3], epsilon [MeV/fm^3], n_B [fm^-3]\n"
    header += f"# {len(nB)} points"
    
    data = np.column_stack([P, epsilon, nB])
    np.savetxt(filename, data, header=header, fmt='%.8e',
               delimiter='\t', comments='')
    
    print(f"Saved: {filename} ({len(nB)} points)")
    print(f"  nB: [{nB.min():.4f}, {nB.max():.4f}] fm^-3")
    print(f"  P:  [{P.min():.2f}, {P.max():.2f}] MeV/fm^3")
    print(f"  e:  [{epsilon.min():.2f}, {epsilon.max():.2f}] MeV/fm^3")

## Compute TOV

In [ ]:
# ==============================================================================
# TOV COMPUTATION (using compute_tov_sequence)
# ==============================================================================
from eos.astro.tov.solver import compute_tov_sequence, generate_ec_logspace
from pathlib import Path
import matplotlib.pyplot as plt

#from eos import REPO_ROOT
#OUTDIR_TOV = REPO_ROOT / "output" / "zlvmit" / "tov"
OUTDIR_TOV = Path(DIR) / "tov"
os.makedirs(OUTDIR_TOV , exist_ok=True)

# Central energy density range
e_c_vec = generate_ec_logspace(100.0, 2500.0, 80)

# Store results
tov_results = {}

for eta in eta_values:
    eos_file = OUTDIR_TOV / f"eos_T0_betaeq_eta{eta:.2f}.dat"
    output_file = OUTDIR_TOV / f"tov_eta{eta:.2f}.dat"
    
    tov_results[eta] = compute_tov_sequence(
        eos_input=str(eos_file),
        skip_header=3,
        e_c_vec=e_c_vec,
        add_crust_table='BPS',
        add_crust_mode='interpolate',
        n_transition=0.08,
        delta_n=0.016,  # 0.1 * n0
        compute_baryonic_mass=True,
        compute_tidal=False,
        output_file=str(output_file),
        verbose=True,
    )
    print(f"η={eta}: {len(tov_results[eta])} stars computed\n")




In [ ]:
# ==============================================================================
# PLOTTING (2x2 layout with P vs ε)
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for eta in eta_values:
    data = tov_results[eta]
    # Columns: e_c, n_c, P_c, R, M, M_b
    e_c, n_c, P_c, R, M, M_b = data[:, 0], data[:, 1], data[:, 2], data[:, 3], data[:, 4], data[:, 5]
    
    style = ETA_STYLES[eta]
    axes[0, 0].plot(R, M, color=style['color'], ls=style['linestyle'], 
                    lw=style['linewidth'], label=style['label'])
    axes[0, 1].plot(M, M_b, color=style['color'], ls=style['linestyle'], 
                    lw=style['linewidth'], label=style['label'])
    axes[1, 0].semilogx(e_c, M, color=style['color'], ls=style['linestyle'], 
                        lw=style['linewidth'], label=style['label'])
    axes[1, 1].plot(e_c, P_c, color=style['color'], ls=style['linestyle'], 
                      lw=style['linewidth'], label=style['label'])

# M-R plot
axes[0, 0].set_xlabel(r'$R$ [km]')
axes[0, 0].set_ylabel(r'$M$ [$M_\odot$]')
axes[0, 0].set_xlim(10, 14)
axes[0, 0].set_ylim(0.1, 2.5)
axes[0, 0].legend(loc='lower left')
axes[0, 0].grid(True, alpha=0.3)

# M vs M_b plot
axes[0, 1].set_xlabel(r'$M$ [$M_\odot$]')
axes[0, 1].set_ylabel(r'$M_b$ [$M_\odot$]')
axes[0, 1].set_xlim(0, 3.0)
axes[0, 1].set_ylim(0, 3.0)
axes[0, 1].legend(loc='upper left')
axes[0, 1].grid(True, alpha=0.3)

# M vs e_c plot
axes[1, 0].set_xlabel(r'$\varepsilon_c$ [MeV/fm$^3$]')
axes[1, 0].set_ylabel(r'$M$ [$M_\odot$]')
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, alpha=0.3)

# P vs ε plot (central values)
axes[1, 1].set_xlabel(r'$\varepsilon_c$ [MeV/fm$^3$]')
axes[1, 1].set_ylabel(r'$P_c$ [MeV/fm$^3$]')
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTDIR_TOV / 'tov_all_eta_2x2.png', dpi=150, bbox_inches='tight')
plt.show()

# Print M_max summary
print("\nMaximum Mass Summary:")
print("-" * 50)
for eta in eta_values:
    data = tov_results[eta]
    M_max = data[:, 4].max()
    idx = data[:, 4].argmax()
    R_max = data[idx, 3]
    e_c_max = data[idx, 0]
    print(f"  η={eta:.2f}: M_max = {M_max:.3f} M_sun, R = {R_max:.2f} km, ε_c = {e_c_max:.0f} MeV/fm³")
